In [ ]:
import os
import datetime
import colormaps
from pathlib import Path
from functools import partial
from itertools import product
from string import ascii_lowercase
from tqdm import tqdm, trange
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
from matplotlib.colors import (
    LinearSegmentedColormap,
    BoundaryNorm,
    Normalize,
    rgb_to_hsv,
    hsv_to_rgb,
)
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import xarray as xr
import polars as pl
import polars.selectors as cs

os.environ["PATH"] = os.environ["PATH"] + ":/opt/homebrew/bin"

from jetutils.definitions import (
    PRETTIER_VARNAME,
    YEARS,
    UNITS,
    RESULTS,
    FACTORS_UNITS,
    FACTORS,
    DATADIR,
    DUNCANS_REGIONS_NAMES,
    MONTH_NAMES,
    FIGURES,
    RADIUS,
    get_region,
    squarify,
    polars_to_xarray,
    xarray_to_polars,
    to_expr,
    get_index_columns,
)
from jetutils.data import standardize, standardize_polars_dtypes, compute_anomalies_pl
from jetutils.geospatial import (
    central_diff,
    haversine_from_dl,
    compute_relative_clim,
    compute_relative_sm,
    compute_relative_std,
    compute_relative_anom,
    create_all_relative_plots,
    prepare_last_step_1,
    prepare_last_step_2
)
from jetutils.jet_finding import (
    average_jet_categories,
    track_jets,
    slowness_from_cross,
    spells_from_cross,
    connected_from_cross,
    slowness_expr,
    spells_from_cross_catd_simple,
    extract_features,
    gaussian_smooth_func,
    find_all_jets,
    jet_position_as_da,
    to_one_large,
)
from jetutils.plots import (
    STYLE_SHEET,
    COLORS,
    COLORS_EXT,
    WERNLI_FLAIR,
    WERNLI_FLAIR_LEVELS,
    Clusterplot,
    plot_interp,
    plot_relative_time,
    gaussian_kde,
    props_vs_quantiles,
)
from jetutils.anyspell import (
    make_daily,
    mask_from_spells_pl,
    subset_around_onset,
    extend_spells,
    get_spells,
)
from jetutils.stats import (
    create_bootstrapped_times,
    odds_ratio,
    is_signif_OR,
    common_OR,
)

%load_ext IPython.extensions.autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

ALL_TIMES = (
    pl.datetime_range(
        start=pl.datetime(1969, 1, 1),
        end=pl.datetime(2021, 1, 1),
        closed="left",
        interval="6h",
        eager=True,
        time_unit="ms",
    )
    .rename("time")
    .to_frame()
)
summer_filter = ALL_TIMES.filter(pl.col("time").dt.month().is_in([6, 7, 8, 9]))
summer = summer_filter["time"]
summer_daily = summer.filter(summer.dt.hour() == 0)

summer_doy = summer_daily.dt.ordinal_day().unique()
n_bootstraps = 100

RUNS = ["ctrl", "dobl"]
JETS = ["STJ", "EDJ"]
JET_COLORS = [COLORS[2], COLORS[1]]

In [ ]:
both_props = {}
both_cross = {}
both_summary_comps = {}
both_summer_summary_comps = {}
both_spells = {}
both_paths = {}
dist_threshold = 1.2e6
overlap_threshold = 0.6
dis_polar_thresh = 0.12
force_summary_comp = True
for run in RUNS:
    basepath = Path(DATADIR, "Henrik_data", run)
    jets = pl.read_parquet(basepath.joinpath("jets.parquet"))
    cross_opath = basepath.joinpath("cross.parquet")
    if cross_opath.is_file():
        cross = pl.read_parquet(cross_opath)
    else:
        cross = track_jets(jets)
        cross.write_parquet(cross_opath)   
    both_cross[run] = cross
    props = pl.read_parquet(basepath.joinpath("props.parquet"))
    summary_comp_file = basepath.joinpath("summary_comp.parquet")
    if summary_comp_file.is_file() and not force_summary_comp:
        summary_comp = pl.read_parquet(summary_comp_file)
    else:
        _, summary_comp = connected_from_cross(jets, cross, dist_threshold, overlap_threshold, dis_polar_thresh)
        jet_column = (
            pl.when((pl.col("is_polar") > 0.5).mean().over("spell") > 0.5)
            .then(pl.lit("EDJ"))
            .otherwise(pl.lit("STJ"))
        )
        summary_comp = (
            summary_comp
            .drop("s", "theta", "is_polar", "s_right", "theta_right", "is_polar_right")
            .join(props["time", "jet ID", "is_polar"], on=("time", "jet ID"))
            .with_columns(
                jet=jet_column,
                slowness=slowness_expr(),
            )
            .with_columns(
                slowness_sum=pl.col("slowness").sum().over("spell"),
            )
            .sort("time", "jet ID")
        )
        summary_comp.write_parquet(summary_comp_file)
    props = props.join(summary_comp["time", "jet ID", "slowness", "slowness_sum"], on=["time", "jet ID"], how="left")
    both_props[run] = props
    spells_list = spells_from_cross(
        jets,
        cross,
        summary_comp,
        dist_threshold,
        overlap_threshold,
        dis_polar_thresh,
        n_STJ=30,
        n_EDJ=30,
        season=summer,
        STJ_lat_threshold=30,
    )
    
    both_spells[run] = spells_list
    for jet_name, spells in spells_list.items():
        print(run, jet_name, spells["spell"].n_unique())
        print(run, jet_name, spells["len"].min() / 4)
        print(run, jet_name, spells["slowness_sum"].min())
        both_spells[f"{run}_{jet_name}"] = spells
    summary_comp = summary_comp.filter(pl.col("len") > 1)
    summer_summary_comp = summary_comp.filter(
        pl.col("time")
        .is_in(pl.lit(summer.implode().first(), pl.List(pl.Datetime("ms"))))
        .over("spell")
    )
    both_summary_comps[run] = summary_comp
    both_summer_summary_comps[run] = summer_summary_comp
    # both_phat_props[run] = pl.read_parquet(basepath.joinpath("props_full.parquet"))
    # cross_catd = pl.read_parquet(basepath.joinpath("cross.parquet"))
    # break

# more stuff

In [ ]:
filters_for_variables = {
    "AAVO": ["cold", "warm"],
    "CAVO": ["cold", "warm"],
    "t_low": ["cold", "warm"],
    "z500": ["cold", "warm"],
    "DTCOND500": [
        "warm_entrance",
        "cold_exit",
        "cold",
        "warm",
        "warm_far_entrance",
    ],
    "PTTEND500": [
        "warm_entrance",
        "cold_exit",
        "cold",
        "warm",
        "warm_far_entrance",
    ],
    "theta300": [
        "cold_exit",
        "warm_exit",
        "cold_entrance",
        "warm_entrance",
        "cold",
        "warm",
    ],
    "theta300:grad": [
        "core",
    ],
    "zeta300": [
        "cold_exit",
        "warm_exit",
        "cold_entrance",
        "warm_entrance",
        "cold",
        "warm",
    ],
    "hor": ["cold", "warm", "core"],
    "vert": ["cold", "warm", "core"],
}
reduce_dict = {
    var: pl.Expr.any for var in ["AAVO", "CAVO"]
}

for run in RUNS:
    basepath = Path(DATADIR, "Henrik_data", run)
    pwe_path = basepath.joinpath("props_with_extras.parquet")
    if not pwe_path.is_file():
        props_with_extras = prepare_last_step_1(
            basepath, filters_for_variables, both_props[run], reduce_dict
        )
        props_with_extras.write_parquet(pwe_path)
    else:
        props_with_extras = pl.read_parquet(pwe_path)

    everything = {}
    for jet_name in ["STJ", "EDJ"]:
        everything[f"{run}_{jet_name}"] = prepare_last_step_2(
            props_with_extras,
            both_spells[f"{run}_{jet_name}"],
            summer,
            grams_wr=None,
            n_bootstraps=1000,
        )
        for when, maybe_other in product(["before", "during"], ["", "-other"]):
            col1 = pl.col(f"AAVO-warm.{when}{maybe_other}")
            col2 = pl.col(f"CAVO-cold.{when}{maybe_other}")
            col3 = f"DPVOany.{when}{maybe_other}"
            everything[f"{run}_{jet_name}"] = everything[f"{run}_{jet_name}"].with_columns(**{col3: col1 + col2})

# stats

In [ ]:
figure = plt.figure(figsize=(12, 9), constrained_layout=True)
subfigs = figure.subfigures(2, 2)
subfigs = subfigs.ravel()

def annotate(ax, annotation, coords=None):
    if coords is None:
        coords = (0.97, 0.97)
    ax.annotate(
        annotation,
        coords,
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontweight="demi",
        bbox={
            "boxstyle": "square, pad=0.1",
            "edgecolor": "none",
            "facecolor": "white",
        },
        usetex=False,
    )
    return ax


# distances, slowness_sum, lifetime lengths, episode distrib
jet_name_expr = (
    pl.when(pl.col("is_polar") < 0.5).then(pl.lit("STJ")).otherwise(pl.lit("EDJ"))
)
colors = [COLORS[2], COLORS[1]]
letters_list = np.split(np.array(list(ascii_lowercase))[:24], 6)

## cross distances, no smoothing
fig = subfigs[0]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[0]
for run, axs, otheraxs, letters_ in zip(RUNS, axes, axes[::-1], np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    cross_summer = summer_filter.join(both_cross[run], on="time")
    cross_summer = cross_summer.filter(pl.col("dis_polar") < dis_polar_thresh).with_columns(
        jet=jet_name_expr
    )
    for jet_name, color, ax, otherax, letter in zip(JETS, colors, axs, otheraxs, letters_):
        df_ = cross_summer.filter(pl.col("jet") == jet_name)
        ax.hist(
            df_["dist"] / 1e6,
            color=color,
            bins=np.linspace(0, 10, 31),
            density=True,
            edgecolor="white",
            linewidth=1,
        )
        otherax.hist(
            df_["dist"] / 1e6,
            color="black",
            bins=np.linspace(0, 10, 31),
            density=True,
            histtype="step",
        )
        ax.axvline(dist_threshold / 1e6, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Dist. between jets [1000 km]")
fig.supylabel("Distance density")
axes[1, 0].set_xticks(np.arange(11))

## Persistence
fig = subfigs[1]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[1]
for run, axs, otheraxs, letters_ in zip(RUNS, axes, axes[::-1], np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    for jet_name, color, ax, otherax, letter in zip(JETS, colors, axs, otheraxs, letters_):
        df_ = (
            df.group_by("spell")
            .agg(pl.col("slowness_sum").first(), pl.col("jet").first())
            .filter(pl.col("jet") == jet_name)
        )
        ax.hist(
            df_["slowness_sum"],
            color=color,
            bins=np.linspace(0, 4, 41),
            density=True,
            log=True,
            edgecolor="white",
            linewidth=1,
        )
        otherax.hist(
            df_["slowness_sum"],
            color="black",
            bins=np.linspace(0, 4, 41),
            density=True,
            histtype="step",
        )
        cutoff = both_spells[f"{run}_{jet_name}"]["slowness_sum"].min()
        ax.axvline(cutoff, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Persistence [s/m]")
fig.supylabel("Persistence density")
axes[1, 0].set_xticks(np.linspace(0, 4, 9))

## Lengths
fig = subfigs[2]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[2]
for run, axs, otheraxs, letters_ in zip(RUNS, axes, axes[::-1], np.split(letters, 2)):
    df = both_summer_summary_comps[run]
    for jet_name, color, ax, otherax, letter in zip(JETS, colors, axs, otheraxs, letters_):
        df_ = (
            df.group_by("spell")
            .agg(pl.col("len").first(), pl.col("jet").first())
            .filter(pl.col("jet") == jet_name)
        )
        ax.hist(
            df_["len"] / 4,
            color=color,
            bins=np.arange(2, 10, 0.25) - 0.125,
            density=False,
            log=True,
            edgecolor="white",
            linewidth=1,
        )
        otherax.hist(
            df_["len"] / 4,
            color="black",
            bins=np.arange(2, 10, 0.25) - 0.125,
            density=False,
            histtype="step",
        )
        cutoff = both_spells[f"{run}_{jet_name}"]["len"].min() / 4
        ax.axvline(cutoff, color="black", ls="dashed")
        ax = annotate(ax, letter)
fig.supxlabel("Lifetime [day]")
fig.supylabel("Lifetime bin count")
axes[1, 0].set_xticks(np.arange(1, 10))

## When PEs
fig = subfigs[3]
axes = fig.subplots(2, 2, sharex="all", sharey="all")
axes = axes.T
letters = letters_list[3]
bins = pl.datetime_range(
    datetime.datetime(1959, 6, 1),
    datetime.datetime(1959, 10, 1),
    "7d",
    closed="left",
    eager=True,
)
for run, axs, otheraxs, letters_ in zip(RUNS, axes, axes[::-1], np.split(letters, 2)):
    huh = (
        summer.dt.ordinal_day()
        .unique()
        .sort()
        .to_frame()
        .with_columns(
            time_2=pl.datetime(year=1959, month=1, day=1) + pl.duration(days="time")
        )
    )
    for jet_name, color, ax, otherax, letter in zip(JETS, colors, axs, otheraxs, letters_):
        df_ = both_spells[f"{run}_{jet_name}"].select(
            time=pl.datetime(year=1959, month=1, day=1, time_unit="ms")
            + (
                pl.col("time")
                - pl.datetime(year=pl.col("time").dt.year(), month=1, day=1, time_unit="ms")
            )
        )
        ax.hist(df_["time"], bins=bins, color=color, edgecolor="white", linewidth=1)
        otherax.hist(df_["time"], bins=bins, color="black", histtype="step")
        ax = annotate(ax, letter)

    ax.xaxis.set_major_formatter(DateFormatter("%m-%d"))
    ax.xaxis.set_tick_params(labelsize=11, rotation=30)
    ticks = ax.get_xticks()
    ticklabels = ax.get_xticklabels()
    ax.set_xticks(ticks, labels=[t.get_text() for t in ticklabels], ha="right")
fig.supylabel("Episode bin count")

# figure.savefig(f"{FIGURES}/ClimateChange/stats.pdf")

# mean props

In [ ]:
from matplotlib.dates import MonthLocator
from jetutils.data import periodic_rolling_pl
from matplotlib.lines import Line2D
data_vars: list = ["mean_lat", "mean_s", "mean_theta", "wavinessR16", "slowness", "slowness_sum"]
nrows: int = 2
ncols: int = 3

plot_kwargs = {"ctrl": [average_jet_categories(both_props["ctrl"]), "solid"], "dobl": [average_jet_categories(both_props["dobl"]), "dashed"]}

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 3.5, nrows * 2.4),
    constrained_layout=True,
    sharex="all",
)
axes = axes.flatten()
jets = average_jet_categories(both_props["ctrl"])["jet"].unique().to_numpy()
njets = len(jets)
gb_args = [pl.col("time").dt.ordinal_day().alias("dayofyear"), pl.col("jet")]
aggs = [
    pl.col(col).replace([float("-inf"), float("inf"), float("nan")], None).mean()
    for col in data_vars
]


for name, args in plot_kwargs.items():
    props, ls = args
    gb = props.group_by(gb_args)

    means = gb.agg(aggs).sort("dayofyear", "jet", descending=[False, True])
    means = squarify(means, ["dayofyear", "jet"])
    means = periodic_rolling_pl(means, 15, data_vars)

    x = means["dayofyear"].unique()
    color_order = [2, 1]
    for letter, varname, ax in zip(ascii_lowercase, data_vars, axes.ravel()):
        dji = varname == "double_jet_index"
        factor = FACTORS.get(varname, 1)
        if factor == 1:
            factor_str = ""
        else:
            factor_str = str(int(np.log10(np.abs(factor))))
            factor_str = r"$10^{" + factor_str + r"} \times $"
        ys = means[varname].to_numpy().reshape(366, njets)
        for i in range(njets):
            factor = factor / (1 + 9 * (varname in ["slowness", "slowness_sum"] and i == 1))
            color = "black" if dji else COLORS[color_order[i]]
            ax.plot(
                x,
                ys[:, i] / factor,
                lw=3,
                color=color,
                zorder=10,
                ls=ls,
            )
            if dji:
                break
        ax.set_title(
            f"{letter}) {PRETTIER_VARNAME.get(varname, varname)} [{factor_str}{UNITS.get(varname, '')}]"
        )
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
        ax.set_xlim(min(x), max(x))
        if varname == "mean_lev":
            ax.invert_yaxis()
        # if name == "dobl":
        #     ylim = ax.get_ylim()
        #     wherex = np.isin(x, JJASDOYS)
        #     ax.fill_between(
        #         x, *ylim, where=wherex, alpha=0.1, color="black", zorder=-10
        #     )
        #     ax.set_ylim(ylim)
           
linedata =  [[0, 1], [0, 1]]
handles = [
    Line2D(*linedata, lw=2, ls="solid", color=COLORS[2]),
    Line2D(*linedata, lw=2, ls="solid", color=COLORS[1]),
    Line2D(*linedata, lw=2, ls="solid", color="black"),
    Line2D(*linedata, lw=2, ls="dashed", color="black"),
]
labels = [*JETS, *RUNS]
axes.ravel()[1].legend(ncol=2, handles=handles, labels=labels, labelspacing=0.2, handletextpad=0.2, columnspacing=0.01, handlelength=1.1).set_zorder(102)
# fig.savefig(f"{FIGURES}/Diabatic/seasonal.pdf")

In [ ]:
diffs = (
    average_jet_categories(both_props["ctrl"])
    .group_by("jet", dayofyear=pl.col("time").dt.ordinal_day())
    .agg(cs.numeric().mean().cast(pl.Float32()))
    .join(
        average_jet_categories(both_props["dobl"])
        .group_by("jet", dayofyear=pl.col("time").dt.ordinal_day())
        .agg(cs.numeric().mean().cast(pl.Float32())),
        on=["dayofyear", "jet"],
        suffix="_dobl"
    )
    .select("jet", "dayofyear", *[(pl.col(f"{col}_dobl") - pl.col(col)).alias(col) for col in both_props["ctrl"].columns if col not in ["time", "jet ID"]])
)
means = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").mean()).sort("jet", "dayofyear")
stds = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").std()).sort("jet", "dayofyear")
medians = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").median()).sort("jet", "dayofyear")
q1 = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").quantile(0.2)).sort("jet", "dayofyear")
q2 = diffs.group_by("jet", "dayofyear").agg(cs.exclude("member", "member_future").quantile(0.8)).sort("jet", "dayofyear")
for var in ["mean_lat", "mean_s", "mean_theta", "wavinessR16", "slowness", "slowness_sum"]:
    plt.figure()
    for jet, color in zip(JETS, JET_COLORS):
        means_ = means.filter(pl.col("jet") == jet)
        stds_ = stds.filter(pl.col("jet") == jet)
        q1_ = q1.filter(pl.col("jet") == jet)
        q2_ = q2.filter(pl.col("jet") == jet)
        medians_ = medians.filter(pl.col("jet") == jet)
        plt.fill_between(means_["dayofyear"], q1_[var], q2_[var], alpha=0.2, color=color)
        plt.plot(means_["dayofyear"], means_[var], color=color, lw=4)
        plt.plot(means_["dayofyear"], medians_[var], color=color, ls="dashed")
    plt.suptitle(var)

# interp around

In [ ]:
variables = [
    "theta300",
    "theta300:grad",
    "s300",
    "zeta300",
    "z500",
    "t_low",
    "DTCOND500",
    "PTTEND500",
    "AAVO",
    "CAVO",
    "hor",
    "vert",
]
for run in RUNS:
    create_all_relative_plots(
        {jet: both_spells[f"{run}_{jet}"] for jet in JETS},
        variables,
        basepath = Path(DATADIR, "Henrik_data", run),
        odir=Path(FIGURES, "Diabatic/figure_data", run),
        season=summer,
        n_bootstraps=100,
    )

In [ ]:
variables = {
    "hor:clim": [8, colormaps.BlWhRe, [-400, 400]],
    "hor:anom": [8, colormaps.BlWhRe, [-200, 200]],
    "vert:clim": [8, colormaps.BlWhRe, [-400, 400]],
    "vert:anom": [8, colormaps.BlWhRe, [-300, 300]],
    "vert_extra:clim": [8, colormaps.BlWhRe, [-400, 400]],
    "vert_extra:anom": [8, colormaps.BlWhRe, [-300, 300]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/EPF")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "AAVO:clim": [8, colormaps.amp, [0, 40]],
    "AAVO:anom": [8, colormaps.BlWhRe, [-10, 10]],
    "CAVO:clim": [8, colormaps.amp, [0, 40]],
    "CAVO:anom": [8, colormaps.BlWhRe, [-10, 10]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/RWB")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "theta300:clim": [9, colormaps.amp, [310, 350]],
    "theta300:anom": [8, colormaps.BlWhRe, [-5, 5]],
    "theta300:clim_grad": [8, colormaps.bilbao_r, [0, 20]],
    "theta300:anom_grad": [8, colormaps.BlWhRe, [-10, 10]],
    "DTCOND500:clim": [8, colormaps.amp, [0, 4.0]],
    "DTCOND500:anom": [8, colormaps.BlWhRe, [-2.0, 2.0]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/theta")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "z500:clim": [7, colormaps.lipari_r, [5400, 6000]],
    "z500:anom": [8, colormaps.BlWhRe, [-100, 100]],
    "t_low:clim": [9, colormaps.amp, [270, 310]],
    "t_low:anom": [8, colormaps.BlWhRe, [-5, 5]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/meteo")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "s300:clim": [8, colormaps.amp, [0, 70]],
    "s300:anom": [8, colormaps.BlWhRe, [-8, 8]],
    "zeta300:clim": [8, colormaps.amp, [0, 20]],
    "zeta300:anom": [8, colormaps.BlWhRe, [-3, 3]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/wind")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()
    
for exp_, jet in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    fig.savefig(odir.joinpath(f"{exp_}_{jet}.pdf"))
    # plt.close()

# differences

In [ ]:
from jetutils.geospatial import create_all_relative_diff_plots
variables = [
    "theta300",
    "theta300:grad",
    "s300",
    "zeta300",
    "z500",
    "t_low",
    "DTCOND500",
    "PTTEND500",
    "AAVO",
    "CAVO",
    "hor",
    "vert",
]
create_all_relative_diff_plots(
    both_spells,
    variables,
    runs=RUNS,
    basepath = Path(DATADIR, "Henrik_data"),
    odir=Path(FIGURES, "Diabatic/figure_data", "diff"),
    season=summer,
)

In [ ]:
variables = {
    "hor:clim": [8, colormaps.BlWhRe, [-400, 400]],
    "hor:anom": [8, colormaps.BlWhRe, [-400, 400]],
    "vert:clim": [8, colormaps.BlWhRe, [-400, 400]],
    "vert:anom": [8, colormaps.BlWhRe, [-400, 400]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/EPF")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["diff"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "AAVO:clim": [8, colormaps.BlWhRe, [-15, 15]],
    "AAVO:anom": [8, colormaps.BlWhRe, [-15, 15]],
    "CAVO:clim": [8, colormaps.BlWhRe, [-15, 15]],
    "CAVO:anom": [8, colormaps.BlWhRe, [-15, 15]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/RWB")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["diff"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "theta300:clim": [8, colormaps.BlWhRe, [-3, 3]],
    "theta300:anom": [8, colormaps.BlWhRe, [-3, 3]],
    "theta300:clim_grad": [8, colormaps.BlWhRe, [-2, 2]],
    "theta300:anom_grad": [8, colormaps.BlWhRe, [-4, 4]],
    "DTCOND500:clim": [8, colormaps.BlWhRe, [-4.0, 4.0]],
    "DTCOND500:anom": [8, colormaps.BlWhRe, [-4.0, 4.0]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/theta")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["diff"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "z500:clim": [8, colormaps.BlWhRe, [-100, 100]],
    "z500:anom": [8, colormaps.BlWhRe, [-100, 100]],
    "t_low:clim": [8, colormaps.BlWhRe, [-1.5, 1.5]],
    "t_low:anom": [8, colormaps.BlWhRe, [-3, 3]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/meteo")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["diff"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()

In [ ]:
variables = {
    "s300:clim": [8, colormaps.BlWhRe, [-5, 5]],
    "s300:anom": [8, colormaps.BlWhRe, [-5, 5]],
    "zeta300:clim": [8, colormaps.BlWhRe, [-3, 3]],
    "zeta300:anom": [8, colormaps.BlWhRe, [-3, 3]],
}

ipath = Path(f"{FIGURES}/Diabatic/figure_data")
odir = Path(f"{FIGURES}/Diabatic/wind")
odir.mkdir(exist_ok=True)

for exp_, jet in product(["diff"], ["STJ", "EDJ"]):
    fig = plot_interp(variables, "", ipath.joinpath(exp_), jet, handle_pvals="hatch", n_col=2, square_len=3.3, transpose=True)
    # fig.savefig(odir.joinpath(f"{exp_}_{jet}_bc.pdf"))
    # plt.close()
    

# props

In [ ]:
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    "mean_lon",
    "mean_lat",
    "mean_theta",
    "mean_s",
    "width",
    "waviness1",
    "waviness2",
    "wavinessDC16",
    "wavinessFV15",
    "is_polar",
    "int",
    "int_over_europe",
]


def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()


def mean_confidence(col: pl.Series, q: float) -> pl.Series:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    mu = col.mean()
    s_sq = col.var()
    if s_sq is None:
        return None
    s = np.sqrt(s_sq)
    sign = 1 - 2 * int(q < 0.5)
    q = q if q < 0.5 else 1 - q
    if n > 10:
        to_ret = mu + sign * np.abs(norm.ppf(q=q)) / np.sqrt(n) * s
    else:
        to_ret = mu + sign * s / np.sqrt(n) * t.ppf(q=1 - q, df=n - 1)
    to_ret = np.clip(to_ret, mu - 5 * s, mu + 5 * s)
    return to_ret


def var_confidence(col: pl.Series, q) -> float:
    n = (col.is_not_null() & col.is_not_nan()).sum()
    s_sq = col.var()
    if s_sq is None:
        return None
    sign = 1 - 2 * int(q < 0.5)
    if n > 50:
        q = q if q < 0.5 else 1 - q
        to_ret = s_sq + sign * np.sqrt(2 / n) * np.abs(norm.ppf(q)) * s_sq
    else:
        to_ret = (n - 1) * s_sq / chi2.ppf(1 - q, df=n - 1)
    return np.clip(to_ret, 0, s_sq * 2)


def func_q(col, q):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).map_batches(
            partial(var_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
        )
    return pl.col(col).map_batches(
        partial(mean_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
    )


q_mean = 1e-15
for exp_, spell_of in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    spells_from_jet = both_spells[f"{exp_}_{spell_of}"].drop("slowness_sum")
    n_spells = spells_from_jet["spell"].n_unique()
    props = average_jet_categories(both_props[exp_])
    props_masked = mask_from_spells_pl(
        spells_from_jet, props, time_before=datetime.timedelta(days=4)
    )
    props_masked = props_masked.filter(
        pl.col("spell").n_unique().over("relative_index") > 14
    )
    aggs = {col: func(col) for col in data_vars}
    aggs = aggs | {f"{col}_10": func_q(col, q_mean) for col in data_vars}
    aggs = aggs | {f"{col}_90": func_q(col, 1 - q_mean) for col in data_vars}
    explode_list = [f"{col}_10" for col in data_vars] + [
        f"{col}_90" for col in data_vars
    ]
    aggs = aggs | {"alive": pl.col("time").len()}
    mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs
    )
    aggs_ = {col: func_q(col, 0.95) for col in data_vars}
    q25 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    aggs_ = {col: func_q(col, 0.05) for col in data_vars}
    q75 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    fig, axes = plt.subplots(3, 4, figsize=(11, 8), tight_layout=False, sharex="all")
    axes = axes.ravel()
    props_summer = summer_filter.join(props, on="time")
    means = props_summer.group_by("jet", maintain_order=True).agg(**aggs)
    alive_spells = (
        props_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    for j, jet in enumerate(["STJ", "EDJ"]):
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        q25_ = q25.filter(pl.col("jet") == jet)
        q75_ = q75.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=COLORS[2 - j], lw=2.5)
            ax.fill_between(
                x,
                q25_[data_var] / factor,
                q75_[data_var] / factor,
                color=COLORS[2 - j],
                alpha=0.4,
            )
            mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
            q10 = means.filter(pl.col("jet") == jet)[f"{data_var}_10"].item() / factor
            q90 = means.filter(pl.col("jet") == jet)[f"{data_var}_90"].item() / factor
            ax.plot([x[0], x[-1]], [mean, mean], color=COLORS[2 - j], ls="dashed", lw=2)
            if j == 0:
                factor_str = (
                    "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                )
                ax.set_title(
                    rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    for i, ax in enumerate(axes):
        ax.axvline(0, zorder=1, color="black", lw=2)
        if i > 11:
            ax.set_xlabel("Relative time around onset [days]", color="black")
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        ybounds = [
            ylim[0] - 0.05 * (ylim[1] - ylim[0]),
            ylim[0] + 0.05 * (ylim[1] - ylim[0]),
        ]
        im = ax.pcolormesh(
            x,
            ybounds,
            alive_spells[None, :-1],
            zorder=-10,
            cmap=colormaps.greys,
            alpha=0.7,
            vmin=0,
        )
    fig.set_constrained_layout(True)
    fig.suptitle(
        f"Persistent episodes of the {spell_of[:3]}. Exp: {exp_}. {props_masked['spell'].n_unique()} spells"
    )
    fig.savefig(f"{FIGURES}/Diabatic/{exp_}_{spell_of}_props.pdf")
    # plt.close()

In [ ]:
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    "AAVO-warm",
    "AAVO-cold",
    "CAVO-warm",
    "CAVO-cold",
    "t_low-warm",
    "t_low-cold",
    "z500-warm",
    "z500-cold",
    "DTCOND500-warm",
    "DTCOND500-warm_entrance",
    "DTCOND500-cold",
    "PTTEND500-warm",
    "PTTEND500-cold",
    "hor-core",
    "vert-core",
    "theta300:grad-core",
]


def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()


def func_q(col, q):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).map_batches(
            partial(var_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
        )
    return pl.col(col).map_batches(
        partial(mean_confidence, q=q), return_dtype=pl.Float64(), returns_scalar=True
    )


q_mean = 1e-15
for exp_, spell_of in product(["ctrl", "dobl"], ["STJ", "EDJ"]):
    spells_from_jet = both_spells[f"{exp_}_{spell_of}"].drop("slowness_sum")
    n_spells = spells_from_jet["spell"].n_unique()
    props = Path(DATADIR, "Henrik_data", exp_, "props_with_extras.parquet")
    props = pl.read_parquet(props)
    props = average_jet_categories(props)
    props_masked = mask_from_spells_pl(
        spells_from_jet, props, time_before=datetime.timedelta(days=4)
    )
    props_masked = props_masked.filter(
        pl.col("spell").n_unique().over("relative_index") > 14
    )
    aggs = {col: func(col) for col in data_vars}
    aggs = aggs | {f"{col}_10": func_q(col, q_mean) for col in data_vars}
    aggs = aggs | {f"{col}_90": func_q(col, 1 - q_mean) for col in data_vars}
    explode_list = [f"{col}_10" for col in data_vars] + [
        f"{col}_90" for col in data_vars
    ]
    aggs = aggs | {"alive": pl.col("time").len()}
    mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs
    )
    aggs_ = {col: func_q(col, 0.95) for col in data_vars}
    q25 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    aggs_ = {col: func_q(col, 0.05) for col in data_vars}
    q75 = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
        **aggs_
    )
    fig, axes = plt.subplots(4, 4, figsize=(11, 11), tight_layout=False, sharex="all")
    axes = axes.ravel()
    props_summer = summer_filter.join(props, on="time")
    means = props_summer.group_by("jet", maintain_order=True).agg(**aggs)
    alive_spells = (
        props_masked.group_by("relative_index")
        .agg(pl.col("spell").n_unique())
        .sort("relative_index")["spell"]
        .to_numpy()
    )
    for j, jet in enumerate(["STJ", "EDJ"]):
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        q25_ = q25.filter(pl.col("jet") == jet)
        q75_ = q75.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=COLORS[2 - j], lw=2.5)
            ax.fill_between(
                x,
                q25_[data_var] / factor,
                q75_[data_var] / factor,
                color=COLORS[2 - j],
                alpha=0.4,
            )
            mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
            q10 = means.filter(pl.col("jet") == jet)[f"{data_var}_10"].item() / factor
            q90 = means.filter(pl.col("jet") == jet)[f"{data_var}_90"].item() / factor
            ax.plot([x[0], x[-1]], [mean, mean], color=COLORS[2 - j], ls="dashed", lw=2)
            if j == 0:
                factor_str = (
                    "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                )
                ax.set_title(
                    rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    for i, ax in enumerate(axes):
        ax.axvline(0, zorder=1, color="black", lw=2)
        if i > 11:
            ax.set_xlabel("Relative time around onset [days]", color="black")
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        ybounds = [
            ylim[0] - 0.05 * (ylim[1] - ylim[0]),
            ylim[0] + 0.05 * (ylim[1] - ylim[0]),
        ]
        im = ax.pcolormesh(
            x,
            ybounds,
            alive_spells[None, :-1],
            zorder=-10,
            cmap=colormaps.greys,
            alpha=0.7,
            vmin=0,
        )
    fig.set_constrained_layout(True)
    fig.suptitle(
        f"Persistent episodes of the {spell_of[:3]}. Exp: {exp_}. {props_masked['spell'].n_unique()} spells"
    )
    fig.savefig(f"{FIGURES}/Diabatic/{exp_}_{spell_of}_extraprops.pdf")
    # plt.close()

In [ ]:
from jetutils.plots import COLORS_EXT
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    # "mean_lon",
    "mean_lat",
    # "mean_theta",
    "mean_s",
    # "width",
    # "tilt",
    # "waviness1",
    # "waviness2",
    # "wavinessDC16",
    # "wavinessFV15",
    "wavinessR16",
    # "mean_lat_var",
    # "mean_s_var",
    # "is_polar",
    # "int",
    "int_over_europe",
    # "pers",
]

nrows: int = 2
ncols: int = 2

def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()

colors = {
    "ctrl_STJ": COLORS_EXT[6],
    "ctrl_EDJ": COLORS_EXT[3],
    "dobl_STJ": COLORS_EXT[8],
    "dobl_EDJ": COLORS_EXT[5],
}

linestyles = {
    "ctrl_STJ": "solid",
    "ctrl_EDJ": "solid",
    "dobl_STJ": "solid",
    "dobl_EDJ": "solid",
}

for spell_of in ["STJ", "EDJ"]:
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=False, sharex="all")
    axes = axes.ravel()
    for exp_ in ["ctrl", "dobl"]:
        spells_from_jet = both_spells[f"{exp_}_{spell_of}"].drop("slowness_sum")
        n_spells = spells_from_jet["spell"].n_unique()
        props = average_jet_categories(both_props[exp_])
        props_masked = mask_from_spells_pl(
            spells_from_jet, props, time_before=datetime.timedelta(days=4)
        )
        props_masked = props_masked.filter(
            pl.col("spell").n_unique().over("relative_index") > 14
        )
        aggs = {col: func(col) for col in data_vars}
        aggs = aggs | {"alive": pl.col("time").len()}
        mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
            **aggs
        )
        props_summer = summer_filter.join(props, on="time")

        means = props_summer.group_by("jet", maintain_order=True).agg(**aggs)
        alive_spells = (
            props_masked.group_by("relative_index")
            .agg(pl.col("spell").n_unique())
            .sort("relative_index")["spell"]
            .to_numpy()
        )
        j = int(spell_of=="EDJ")
        jet = spell_of
        # for j, jet in enumerate(["STJ", "EDJ"]):
        color = colors[f"{exp_}_{jet}"]
        linestyle = linestyles[f"{exp_}_{jet}"]
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=color, lw=2.5, linestyle=linestyle, label=f"{exp_}")
            mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
            ax.axhline(mean, color=color, ls="dashed", lw=2)
            factor_str = (
                "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
            )
            ax.set_title(
                rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
            )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
        for i, ax in enumerate(axes):
            ax.axvline(0, zorder=1, color="black", lw=2)
            if i > 11:
                ax.set_xlabel("Relative time around onset [days]", color="black")
            ax.set_xlim(x[0], x[-1])
        axes[1].legend(ncol=2, columnspacing=0.5, fontsize=12, loc="upper left")
        fig.set_constrained_layout(True)
        fig.suptitle(f"{n_spells} persistent episodes of the {spell_of}")
        fig.savefig(f"{FIGURES}/Diabatic/oneper_{spell_of}_props.pdf")


In [ ]:
from jetutils.plots import COLORS_EXT
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    # "AAVO-warm",
    # "AAVO-cold",
    # "CAVO-warm",
    # "CAVO-cold",
    # "t_low-warm",
    # "t_low-cold",
    # "z500-warm",
    # "z500-cold",
    # "DTCOND500-warm",
    # "DTCOND500-cold",
    # "PTTEND500-warm",
    # "PTTEND500-cold",
    "DTCOND500-warm_entrance",
    "theta300:grad-core",
    "hor-core",
    "vert-core",
]

nrows: int = 2
ncols: int = 2

def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()

colors = {
    "ctrl_STJ": COLORS_EXT[6],
    "ctrl_EDJ": COLORS_EXT[3],
    "dobl_STJ": COLORS_EXT[8],
    "dobl_EDJ": COLORS_EXT[5],
}

linestyles = {
    "ctrl_STJ": "solid",
    "ctrl_EDJ": "solid",
    "dobl_STJ": "solid",
    "dobl_EDJ": "solid",
}

for spell_of in ["STJ", "EDJ"]:
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=False, sharex="all")
    axes = axes.ravel()
    for exp_ in ["ctrl", "dobl"]:
        spells_from_jet = both_spells[f"{exp_}_{spell_of}"].drop("slowness_sum")
        n_spells = spells_from_jet["spell"].n_unique()
        props = Path(DATADIR, "Henrik_data", exp_, "props_with_extras.parquet")
        props = pl.read_parquet(props)
        props = average_jet_categories(props)
        props_masked = mask_from_spells_pl(
            spells_from_jet, props, time_before=datetime.timedelta(days=4)
        )
        props_masked = props_masked.filter(
            pl.col("spell").n_unique().over("relative_index") > 14
        )
        aggs = {col: func(col) for col in data_vars}
        aggs = aggs | {"alive": pl.col("time").len()}
        mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
            **aggs
        )
        props_summer = summer_filter.join(props, on="time")

        means = props_summer.group_by("jet", maintain_order=True).agg(**aggs)
        alive_spells = (
            props_masked.group_by("relative_index")
            .agg(pl.col("spell").n_unique())
            .sort("relative_index")["spell"]
            .to_numpy()
        )
        j = int(spell_of=="EDJ")
        jet = spell_of
        # for j, jet in enumerate(["STJ", "EDJ"]):
        color = colors[f"{exp_}_{jet}"]
        linestyle = linestyles[f"{exp_}_{jet}"]
        to_plot = mean_ps.filter(pl.col("jet") == jet)
        x = to_plot["relative_index"].unique().to_numpy() / 4
        for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
            factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
            factor = 1e5 if data_var == "width" else factor
            ax.plot(x, to_plot[data_var] / factor, color=color, lw=2.5, linestyle=linestyle, label=f"{exp_}")
            mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
            ax.axhline(mean, color=color, ls="dashed", lw=2)
            factor_str = (
                "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
            )
            ax.set_title(
                rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
            )
            ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
        for i, ax in enumerate(axes):
            ax.axvline(0, zorder=1, color="black", lw=2)
            if i > 11:
                ax.set_xlabel("Relative time around onset [days]", color="black")
            ax.set_xlim(x[0], x[-1])
        axes[1].legend(ncol=2, columnspacing=0.5, fontsize=12, loc="upper left")
        fig.set_constrained_layout(True)
        fig.suptitle(f"{n_spells} persistent episodes of the {spell_of}")
        fig.savefig(f"{FIGURES}/Diabatic/oneper_{spell_of}_extraprops.pdf")


In [ ]:
from jetutils.plots import COLORS_EXT
from string import ascii_lowercase

from scipy.stats import chi2, norm, t

data_vars = [
    # "mean_lon",
    "mean_lat",
    # "mean_theta",
    "mean_s",
    # "width",
    # "tilt",
    # "waviness1",
    # "waviness2",
    # "wavinessDC16",
    # "wavinessFV15",
    "wavinessR16",
    # "mean_lat_var",
    # "mean_s_var",
    # "is_polar",
    # "int",
    "int_over_europe",
    # "pers",
]

nrows: int = 2
ncols: int = 2

def func(col):
    if ":" in col and col.split(":")[-1] == "var":
        return pl.col(col.split(":")[0]).var()
    return pl.col(col).mean()

colors = {
    "ctrl_STJ": COLORS[2],
    "ctrl_EDJ": COLORS[1],
    "dobl_STJ": COLORS_EXT[8],
    "dobl_EDJ": COLORS_EXT[5],
}

linestyles = {
    "ctrl_STJ": "solid",
    "ctrl_EDJ": "solid",
    "dobl_STJ": "solid",
    "dobl_EDJ": "solid",
}

for spell_of in ["STJ", "EDJ"]:
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.4), tight_layout=False, sharex="all")
    axes = axes.ravel()
    for exp_ in ["ctrl", "dobl"]:
        spells_from_jet = both_spells[f"{exp_}_{spell_of}"].drop("slowness_sum")
        n_spells = spells_from_jet["spell"].n_unique()
        props = average_jet_categories(both_props[exp_])
        props_masked = mask_from_spells_pl(
            spells_from_jet, props, time_before=datetime.timedelta(days=4)
        )
        props_masked = props_masked.filter(
            pl.col("spell").n_unique().over("relative_index") > 14
        )
        aggs = {col: func(col) for col in data_vars}
        aggs = aggs | {"alive": pl.col("time").len()}
        mean_ps = props_masked.group_by(["relative_index", "jet"], maintain_order=True).agg(
            **aggs
        )
        props_summer = summer_filter.join(props, on="time")
        means = props_summer.group_by("jet", maintain_order=True).agg(**aggs)
        alive_spells = (
            props_masked.group_by("relative_index")
            .agg(pl.col("spell").n_unique())
            .sort("relative_index")["spell"]
            .to_numpy()
        )
        for j, jet in enumerate(["STJ", "EDJ"]):
            color = colors[f"{exp_}_{jet}"]
            linestyle = linestyles[f"{exp_}_{jet}"]
            to_plot = mean_ps.filter(pl.col("jet") == jet)
            x = to_plot["relative_index"].unique().to_numpy() / 4
            for ax, data_var, letter in zip(axes, data_vars, ascii_lowercase):
                factor = 1e9 if data_var in ["int_over_europe", "int"] else 1
                factor = 1e5 if data_var == "width" else factor
                ax.plot(x, to_plot[data_var] / factor, color=color, lw=2.5, linestyle=linestyle, label=f"{jet}, {exp_}")
                mean = means.filter(pl.col("jet") == jet)[data_var].item() / factor
                ax.axhline(mean, color=color, ls="dashed", lw=2)
                if j == 0:
                    factor_str = (
                        "" if factor == 1 else rf"$10^{int(np.log10(factor))} \times $"
                    )
                    ax.set_title(
                        rf"{letter}) {PRETTIER_VARNAME.get(data_var, data_var)} [{factor_str}{UNITS.get(data_var, '~')}]"
                    )
                ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
        for i, ax in enumerate(axes):
            ax.axvline(0, zorder=1, color="black", lw=2)
            if i > 11:
                ax.set_xlabel("Relative time around onset [days]", color="black")
            ax.set_xlim(x[0], x[-1])
        axes[0].legend(ncol=2, columnspacing=0.5, fontsize=12)
        fig.set_constrained_layout(True)
        fig.suptitle(f"{n_spells} persistent episodes of the {spell_of}")
        fig.savefig(f"{FIGURES}/Diabatic/both_{spell_of}_props.pdf")


# RWB stats

In [ ]:
df = {"ctrl": [], "dobl": []}
for exp_ in ["ctrl", "dobl"]:
    for year in range(1969, 2021):
        f = f"{DATADIR}/Henrik_data/rwb_{exp_}/overturnings_{year}.parquet"
        df[exp_].append(pl.read_parquet(f))
    df[exp_] = pl.concat(df[exp_]).sort("time", "level", "index")
    unique_levs = df[exp_]["level"].unique()
    if len(unique_levs) > 1:
        df[exp_] = df[exp_].filter(pl.col("level") == unique_levs[2])
    df[exp_] = df[exp_].with_columns(pl.col("area") / 1e13)
    df[exp_] = df[exp_].with_columns(
        sum_areas=pl.col("area").sum().over("time", "level"),
        sum_mflux=pl.col("mflux").sum().over("time", "level"),
    )

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 6), sharex="all", sharey="row", constrained_layout=True)
axes = axes.T
colors = {"ctrl": "black", "dobl": "blue", "ctrl_p4": "red"}
prettier_varname = {
    "zeta": r"$\zeta$ [$10^{-5}\mathrm{s^{-1}}$]",
    "mflux": r"$\overline{u'v'}$ [$\mathrm{m^2s^{-2}}$]",
    "sum_mflux": r"$\sum \overline{u'v'}$ [$\mathrm{m^2s^{-2}}$]",
    "area": r"Area [$10^{13}\mathrm{m^{2}}$]",
    "sum_areas": r"Area sum [$10^{13}\mathrm{m^{2}}$]",
}
prettier_orientation = {
    "anticyclonic": "Anticyclonic",
    "cyclonic": "Cyclonic",
}
for orientation, axs in zip(["anticyclonic", "cyclonic"], axes):
    axs[0].set_title(prettier_orientation[orientation])
    for var, ax in zip(["zeta", "mflux", "area"], axs):
        if orientation == "anticyclonic":
            ax.set_ylabel(prettier_varname[var])
        for key, df_ in df.items():
            if key == "ctrl_p4":
                continue
            df_ = df_.drop("points", "side", "geometry").filter(pl.col("orientation") == orientation)
            gb = df_.group_by(pl.col("time").dt.ordinal_day().alias("dayofyear"), "level")
            mean = gb.agg(pl.col(var).mean()).sort("dayofyear", "level")
            mean = periodic_rolling_pl(mean, 31, [var], other_columns=["level"])
            std = gb.agg(pl.col(var).std()).sort("dayofyear", "level")
            std = periodic_rolling_pl(std, 31, [var], other_columns=["level"])
            q25 = gb.agg(pl.col(var).quantile(0.4)).sort("dayofyear", "level")
            q25 = periodic_rolling_pl(q25, 31, [var], other_columns=["level"])
            q75 = gb.agg(pl.col(var).quantile(0.6)).sort("dayofyear", "level")
            q75 = periodic_rolling_pl(q75, 31, [var], other_columns=["level"])
            median = gb.agg(pl.col(var).median()).sort("dayofyear", "level")
            median = periodic_rolling_pl(median, 31, [var], other_columns=["level"])
            # if var != "zeta":
            #     ax.fill_between(mean["dayofyear"], mean[var] - std[var] / 10, mean[var] + std[var] / 10, alpha=0.4, color=colors[key])
                # ax.fill_between(mean["dayofyear"], q25[var], q75[var], alpha=0.4, color=colors[key])
            ax.plot(mean["dayofyear"], mean[var], lw=2.4, color=colors[key], label=key)
            # ax.plot(mean["dayofyear"], median[var], lw=1.4, ls="dashed", color=colors[key])
        ax.xaxis.set_major_locator(MonthLocator(range(0, 13, 3)))
        ax.xaxis.set_major_formatter(DateFormatter("%b"))
axes[0, 0].legend(ncol=2)
fig.savefig(f"{FIGURES}/Diabatic/rwb_props.pdf")